# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Eyram Fenu
**Student ID:** 31122028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [4]:
from google.colab import userdata
from openai import OpenAI

# Retrieve the key securely from Colab Secrets
API_KEY = userdata.get("GROQ_API_KEY")

if not API_KEY:
    raise ValueError(
        "GROQ_API_KEY was not found. "
        "Check Colab Secrets and enable notebook access."
    )

# Groq provides an OpenAI-compatible API
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [5]:
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
    return_usage=False,
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    answer = response.choices[0].message.content

    if return_usage:
        return answer, response.usage

    return answer


answer, usage = ask_llm(
    user_prompt="In two sentences, explain one benefit of microfinance.",
    temperature=0.2,
    max_tokens=100,
    return_usage=True,
)

print("Answer:")
print(answer)

print("\nToken usage:")
print(usage)

Answer:
One benefit of microfinance is that it provides access to financial services for low-income individuals and small business owners who may not have been able to secure traditional loans, allowing them to invest in their businesses and improve their economic stability. By offering small loans and other financial services, microfinance helps to empower entrepreneurs and stimulate local economic growth, ultimately contributing to a reduction in poverty and an improvement in overall well-being.

Token usage:
CompletionUsage(completion_tokens=82, prompt_tokens=52, total_tokens=134, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.00811606, prompt_time=0.002411818, completion_time=0.316762745, total_time=0.319174563)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**
>
> 1. The `system` role gives the model its overall behaviour, responsibilities, and constraints. For example, a system message could say, “You are a careful assistant helping a loan officer. Do not invent missing information.” The `user` role contains the particular question or task the model must address. For example, a user message could say, “Summarize this applicant’s loan request in three sentences.”
>
> 2. A token is a small unit of text processed by a language model. It may be a complete word, part of a word, punctuation, or another text element. API providers bill per token because requests with longer prompts and responses require more processing and computational resources than shorter requests. Token-based billing therefore reflects the approximate amount of work performed for each API call.

### Part 1.2 — Temperature: the randomness dial

In [6]:
question = "Suggest a name for a savings product for market traders in Accra."

print("TEMPERATURE = 0.0")
for run in range(1, 6):
    answer = ask_llm(
        user_prompt=question,
        temperature=0.0,
        max_tokens=100,
    )
    print(f"{run}. {answer}")

print("\nTEMPERATURE = 1.2")
for run in range(1, 6):
    answer = ask_llm(
        user_prompt=question,
        temperature=1.2,
        max_tokens=100,
    )
    print(f"{run}. {answer}")

TEMPERATURE = 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth, which is a key goal for market traders.
3. **Accra Market Fund**: This name is straightforward and clearly communicates the product's purpose and target audience.
4. **
2. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name plays on the idea of saving money being a valuable treasure for market traders.
3. **Sika Saver**: "Sika" is the Ghanaian word for money, so this name incorporates a local touch.

3. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a w

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, the five responses were not completely identical, but they were highly similar. Names such as “Makola Save” and “Trader’s Treasure” appeared repeatedly, showing relatively predictable behaviour. At temperature 1.2, the suggestions were more varied and creative, including names such as “TradeUp Savings,” “Soko Savings,” “SuuKuu,” “Tradera,” and “Sika Safe.”
>
> For a loan decision-support system, I would use a low temperature such as 0.0 or 0.2 because summaries, structured extraction, and risk briefs should be consistent and factual. A higher temperature may generate more varied wording, but it can also increase the risk of unsupported information. For example, some responses claimed that particular words had meanings in Ghanaian languages, but those claims would need independent verification. Therefore, the model’s output should always be reviewed by a human loan officer.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [8]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [9]:
# SUMMARY PROMPT V1: a simple, naive instruction
SUMMARY_PROMPT_V1 = "Summarize this:\n\n{letter_text}"


# SUMMARY PROMPT V2: a clearer role, format, and factual constraints
SUMMARY_SYSTEM_PROMPT_V2 = """
You are an assistant to a microfinance loan officer.

Write a factual and neutral summary of the loan application in 3 to 4 sentences.
Include the applicant's name, requested amount, loan purpose, repayment information,
and relevant financial or security information when stated.

Use only information explicitly contained in the application letter.
Do not guess, infer, exaggerate, or invent missing details.
Clearly indicate when important information is not provided.
""".strip()

SUMMARY_USER_PROMPT_V2 = """
Summarize this loan application:

{letter_text}
""".strip()


def summarize_v1(letter_text):
    return ask_llm(
        user_prompt=SUMMARY_PROMPT_V1.format(letter_text=letter_text),
        temperature=0.7,
        max_tokens=250,
    )


def summarize_v2(letter_text):
    return ask_llm(
        user_prompt=SUMMARY_USER_PROMPT_V2.format(letter_text=letter_text),
        system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
        temperature=0.0,
        max_tokens=250,
    )


# Run both prompt versions on the same two application letters
summary_results = {}

for letter_id in ["L002", "L006"]:
    summary_results[letter_id] = {
        "V1": summarize_v1(LETTERS[letter_id]),
        "V2": summarize_v2(LETTERS[letter_id]),
    }

    print("=" * 80)
    print(f"{letter_id} — SUMMARY V1")
    print("=" * 80)
    print(summary_results[letter_id]["V1"])

    print("\n" + "=" * 80)
    print(f"{letter_id} — SUMMARY V2")
    print("=" * 80)
    print(summary_results[letter_id]["V2"])
    print()

L002 — SUMMARY V1
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when possible, despite not having collateral at the moment.

L002 — SUMMARY V2
Kwame Boateng has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. The purpose of the loan is to address urgent financial needs, with the expectation that his commercial driving business will improve after the festive season. However, the application does not provide specific repayment terms or a detailed repayment schedule. Additionally, Kwame Boateng has stated that he does not have collateral to secure the loan at the moment.

L006 — SUMMARY V1
Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dub

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**
>
> 1. V1 summarized the main facts, but it did not always make missing or uncertain information sufficiently explicit. For L002, V1 said that Kwame “is willing to repay the loan when possible,” whereas V2 clearly stated that “the application does not provide specific repayment terms or a detailed repayment schedule.” V2 therefore presents the repayment uncertainty more directly. For L006, V1 repeated that Kofi was “business-minded” and trustworthy and would repay “when his businesses are successful.” V2 treated these as unsupported claims and added that “No financial information or security details are provided to support the application.” V2 was more neutral and more useful for assessing the application because it distinguished stated claims from documented evidence.
>
> 2. The instruction not to invent details is essential because a loan officer may rely on the summary when assessing risk. If the model adds income, collateral, experience, repayment capacity, or other facts that are absent from the letter, it could unfairly influence the lending decision. This type of failure is called an LLM hallucination: the model generates information that sounds plausible but is unsupported or false. Requiring the model to use only explicitly stated information and identify missing details reduces this risk, although a human loan officer must still verify every summary against the original application.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [10]:
import json
import pandas as pd


# Baseline prompt without the explicit "use null, do not guess" rule.
# This lets us observe why that instruction matters.
EXTRACT_PROMPT_BASELINE = """
Extract the loan application into a JSON object with exactly these keys:

applicant_name
amount_ghs
purpose
monthly_profit_ghs
has_collateral_or_guarantor
repayment_months

Return only the JSON object.

Loan application:
{letter_text}
""".strip()


# Final extraction prompt with an explicit schema, a synthetic few-shot
# example, and instructions for handling missing information.
EXTRACT_PROMPT = """
Extract information from the loan application and return ONLY one valid JSON object.

Use exactly these keys and data types:
{{
  "applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}}

Rules:
- Use only information explicitly stated in the application.
- If a field is not stated, use null. Do not guess.
- Do not calculate or infer missing values.
.
- Do not include explanations, comments, markdown, or extra keys.
- Return valid JSON only.

Example using a fictional application that is not part of the dataset:

Application:
My name is Ama Example. I request GHS 4,000 to purchase baking equipment.
My bakery earns GHS 700 profit monthly. My brother will act as guarantor.
I will repay the loan over 10 months.

Output:
{{
  "applicant_name": "Ama Example",
  "amount_ghs": 4000,
  "purpose": "purchase baking equipment",
  "monthly_profit_ghs": 700,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}}

Now process this application:

{letter_text}
""".strip()


def clean_json_response(response_text):
    """Remove possible Markdown fences before parsing the JSON."""
    cleaned = response_text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    return cleaned.strip()


def extract_fields(letter_text, prompt_template=EXTRACT_PROMPT):
    response_text = ask_llm(
        user_prompt=prompt_template.format(letter_text=letter_text),
        system_prompt=(
            "You are a precise information-extraction system. "
            "Return valid JSON only."
        ),
        temperature=0.0,
        max_tokens=300,
    )

    try:
        cleaned = clean_json_response(response_text)
        return json.loads(cleaned)
    except (json.JSONDecodeError, TypeError) as error:
        print(f"Warning: Could not parse model response as JSON: {error}")
        print("Raw response:", response_text)
        return None


# Test the weaker prompt on a letter with missing financial information.
baseline_result = extract_fields(
    LETTERS["L006"],
    prompt_template=EXTRACT_PROMPT_BASELINE,
)

print("BASELINE RESULT FOR L006")
print(json.dumps(baseline_result, indent=2))


# Run the improved prompt on all six letters.
extraction_results = {}

for letter_id, letter_text in LETTERS.items():
    extraction_results[letter_id] = extract_fields(letter_text)

extraction_df = pd.DataFrame.from_dict(
    extraction_results,
    orient="index",
)

extraction_df.index.name = "letter_id"

print("\nFINAL EXTRACTION RESULTS")
display(extraction_df)

BASELINE RESULT FOR L006
{
  "applicant_name": "Kofi",
  "amount_ghs": 50000,
  "purpose": "start a car washing business, a provision shop, and import phones from Dubai",
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": false,
  "repayment_months": 12
}

FINAL EXTRACTION RESULTS


,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
letter_id,,,,,,
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
>
> 1. The few-shot example should not come from the six letters being processed because that would leak part of the evaluation data into the prompt. The model could copy values or patterns from an actual application instead of demonstrating that it can extract information from unseen letters. A separate fictional example teaches the required JSON structure without contaminating the task.
>
> 2. “Use null, do not guess” tells the model to represent missing information honestly instead of inventing a plausible value. In my baseline test on L006, the model correctly returned `null` for `monthly_profit_ghs` even without the explicit instruction. However, one successful result does not guarantee consistent behaviour across other letters or repeated calls. The explicit rule reduces ambiguity and helps prevent hallucinated financial information from entering the decision-support system.
>
> 3. Temperature 0 is appropriate for structured extraction because the same letter should produce consistent, factual values in the required JSON format. Random variation could change extracted fields, introduce unsupported values, or produce invalid JSON. Creative tasks can benefit from a higher temperature because variation and originality may be desirable, but those qualities are unsafe when extracting facts for a loan assessment.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [11]:
BRIEF_SYSTEM_PROMPT = """
You are a careful decision-support assistant for a microfinance loan officer.

Your role is to organize evidence and suggest follow-up actions. You must not make
the final lending decision. Final decisions are made by qualified human officers.

Use only information contained in the application letter and extracted JSON.
Do not invent, assume, or exaggerate facts.
Do not recommend "approve" or "reject".
""".strip()


BRIEF_PROMPT = """
Prepare a decision-support brief using the application letter and extracted JSON.

Use exactly these headings:

Strengths
- List evidence-based strengths as bullet points.
- If no clear strengths are stated, say so.

Risks / red flags
- List concerns grounded in the supplied information.

Missing information
- List information the loan officer should request or verify.

Suggested next step
- Suggest a non-final action such as inviting the applicant for an interview,
  requesting documents, verifying information, or referring the case for senior review.
- Do not say "approve" or "reject".
- State that the final decision must be made by a human loan officer.

Application letter:
{letter_text}

Extracted JSON:
{extracted_json}
""".strip()


def generate_brief(letter_text, extracted_data):
    return ask_llm(
        user_prompt=BRIEF_PROMPT.format(
            letter_text=letter_text,
            extracted_json=json.dumps(extracted_data, indent=2),
        ),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0.0,
        max_tokens=600,
    )


# Generate a brief for every application
brief_results = {}

for letter_id, letter_text in LETTERS.items():
    brief_results[letter_id] = generate_brief(
        letter_text,
        extraction_results[letter_id],
    )


# Print the three briefs required by the notebook
for letter_id in ["L001", "L002", "L006"]:
    print("=" * 80)
    print(f"{letter_id} — DECISION-SUPPORT BRIEF")
    print("=" * 80)
    print(brief_results[letter_id])
    print()


# Display L003 as well so it can be compared with L006 in the reasoning question
print("=" * 80)
print("L003 — BRIEF FOR REASONING COMPARISON")
print("=" * 80)
print(brief_results["L003"])

L001 — DECISION-SUPPORT BRIEF
## Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
* She has a consistent profit of GHS 900 per month from her current stall, demonstrating a viable income stream.
* Akosua has saved GHS 2,500 through the susu scheme over two years without missing a contribution, showing her ability to manage savings and commitments.
* She has a guarantor, her sister, who is a teacher, potentially providing an additional layer of security for the loan.

## Risks / red flags
* The loan amount of GHS 8,000 is significant compared to her monthly profit of GHS 900, which might pose a risk if her expanded business does not generate enough additional income to cover the loan repayments.
* Expanding into frozen foods with a deep freezer could introduce new operational risks and costs, such as electricity and maintenance expenses, which are not mentioned in the application.

## Missing 

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
>
> 1. The L003 brief is supported by substantially stronger and more verifiable evidence than the L006 brief. L003 identifies a registered business, GHS 5,000 in collateral, GHS 22,000 in annual revenue, GHS 2,800 in average monthly profit, and 18 months of sales records. Its suggested next step focuses on verifying those records, reviewing cash flow, and obtaining information about existing debts. In contrast, L006 proposes three businesses that have not yet started and provides no collateral, guarantor, business records, financial projections, or proven repayment capacity. Its suggested next step therefore requests an interview, detailed business plans, financial projections, and evidence that the proposed ventures are viable.
>
> 2. The brief should support rather than replace the loan officer because an LLM can omit facts, misunderstand statements, introduce unsupported assumptions, or reproduce unfair bias. For example, the L006 brief treated Kofi's enthusiasm and his friends' description of him as possible strengths, even though these are subjective claims rather than verified financial evidence. It also referred to his age as increasing risk, which should not be treated as a lending conclusion without a lawful and relevant policy basis. A human officer must verify the original documents, request missing evidence, apply institutional and legal requirements consistently, and remain accountable for the final decision.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.
> **Commit hash:** 1a1a106

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [12]:
import pandas as pd


EVALUATION_IDS = ["L001", "L003", "L006"]
FIELDS = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]


def values_match(predicted, expected, field):
    # Correctly handle missing values such as None and NaN
    if expected is None:
        return predicted is None or pd.isna(predicted)

    if predicted is None or pd.isna(predicted):
        return False

    # Applicant names may differ only in capitalization or extra spaces
    if field == "applicant_name":
        return str(predicted).strip().casefold() == str(expected).strip().casefold()

    # Purpose is evaluated as case-insensitive text
    if field == "purpose":
        return str(predicted).strip().casefold() == str(expected).strip().casefold()

    # Numeric fields must match exactly
    if field in ["amount_ghs", "monthly_profit_ghs", "repayment_months"]:
        try:
            return float(predicted) == float(expected)
        except (TypeError, ValueError):
            return False

    # Boolean field must match exactly
    if field == "has_collateral_or_guarantor":
        return predicted == expected

    return predicted == expected


evaluation_rows = []

for field in FIELDS:
    row = {"field": field}
    correct_count = 0

    for letter_id in EVALUATION_IDS:
        predicted = extraction_results[letter_id][field]
        expected = GOLD[letter_id][field]
        matched = values_match(predicted, expected, field)

        if matched:
            correct_count += 1

        row[letter_id] = (
            f"{'PASS' if matched else 'FAIL'} | "
            f"predicted: {predicted!r} | gold: {expected!r}"
        )

    row["accuracy"] = correct_count / len(EVALUATION_IDS)
    evaluation_rows.append(row)


accuracy_table = pd.DataFrame(evaluation_rows).set_index("field")
accuracy_table["accuracy"] = accuracy_table["accuracy"].map(
    lambda value: f"{value:.1%}"
)

display(accuracy_table)

,L001,L003,L006,accuracy
field,,,,
applicant_name,PASS | predicted: 'Akosua Mensah' | gold: 'Ako...,PASS | predicted: 'Efua Darko' | gold: 'Efua D...,PASS | predicted: 'Kofi' | gold: 'Kofi',100.0%
amount_ghs,PASS | predicted: 8000 | gold: 8000,PASS | predicted: 15000 | gold: 15000,PASS | predicted: 50000 | gold: 50000,100.0%
purpose,FAIL | predicted: 'buy a deep freezer and expa...,FAIL | predicted: 'purchase two industrial sew...,FAIL | predicted: 'start a car washing busines...,0.0%
monthly_profit_ghs,PASS | predicted: 900 | gold: 900,PASS | predicted: 2800 | gold: 2800,PASS | predicted: None | gold: None,100.0%
has_collateral_or_guarantor,PASS | predicted: True | gold: True,PASS | predicted: True | gold: True,PASS | predicted: False | gold: False,100.0%
repayment_months,PASS | predicted: 20 | gold: 20,PASS | predicted: 15 | gold: 15,PASS | predicted: 12 | gold: 12,100.0%


### Part 4.2 — Reliability: is the system consistent?

In [13]:
def extract_fields_at_temperature(letter_text, temperature):
    response_text = ask_llm(
        user_prompt=EXTRACT_PROMPT.format(letter_text=letter_text),
        system_prompt=(
            "You are a precise information-extraction system. "
            "Return valid JSON only."
        ),
        temperature=temperature,
        max_tokens=300,
    )

    try:
        cleaned = clean_json_response(response_text)
        return json.loads(cleaned)
    except (json.JSONDecodeError, TypeError):
        return None


def run_reliability_test(letter_id, temperature, runs=5):
    results = []

    for run_number in range(1, runs + 1):
        result = extract_fields_at_temperature(
            LETTERS[letter_id],
            temperature=temperature,
        )
        results.append(result)

        print(f"Run {run_number}:")
        print(json.dumps(result, indent=2))
        print()

    valid_results = [result for result in results if result is not None]
    canonical_results = [
        json.dumps(result, sort_keys=True)
        for result in valid_results
    ]

    valid_count = len(valid_results)
    unique_count = len(set(canonical_results))
    all_identical = valid_count == runs and unique_count == 1

    print(f"Valid JSON results: {valid_count}/{runs}")
    print(f"Unique valid results: {unique_count}")
    print(f"All five valid and identical: {all_identical}")

    return results


print("=" * 80)
print("L004 — TEMPERATURE 0.0")
print("=" * 80)
reliability_temp_0 = run_reliability_test(
    letter_id="L004",
    temperature=0.0,
)

print("\n" + "=" * 80)
print("L004 — TEMPERATURE 1.0")
print("=" * 80)
reliability_temp_1 = run_reliability_test(
    letter_id="L004",
    temperature=1.0,
)

L004 — TEMPERATURE 0.0
Run 1:
{
  "applicant_name": "Yaw Owusu",
  "amount_ghs": 12000,
  "purpose": "for feed and 500 new layers",
  "monthly_profit_ghs": 1500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}

Run 2:
{
  "applicant_name": "Yaw Owusu",
  "amount_ghs": 12000,
  "purpose": "for feed and 500 new layers",
  "monthly_profit_ghs": 1500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}

Run 3:
{
  "applicant_name": "Yaw Owusu",
  "amount_ghs": 12000,
  "purpose": "for feed and 500 new layers",
  "monthly_profit_ghs": 1500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}

Run 4:
{
  "applicant_name": "Yaw Owusu",
  "amount_ghs": 12000,
  "purpose": "for feed and 500 new layers",
  "monthly_profit_ghs": 1500,
  "has_collateral_or_guarantor": true,
  "repayment_months": 18
}

Run 5:
{
  "applicant_name": "Yaw Owusu",
  "amount_ghs": 12000,
  "purpose": "for feed and 500 new layers",
  "monthly_profit_ghs": 1500,
  "has_collate

### Part 4.3 — Hallucination probing

In [14]:
# TEST 1 — Ask about information that the application does not contain
hallucination_test_1 = ask_llm(
    user_prompt=f"""
Read the loan application below and answer this question:

What is the applicant's credit score?

If the application does not state the credit score, clearly say that it is not provided.
Do not guess or invent a value.

Application:
{LETTERS["L003"]}
""".strip(),
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0.0,
    max_tokens=150,
)

test_1_pass = (
    "not provided" in hallucination_test_1.lower()
    or "not stated" in hallucination_test_1.lower()
    or "does not provide" in hallucination_test_1.lower()
)

print("=" * 80)
print("TEST 1 — MISSING CREDIT SCORE")
print("=" * 80)
print(hallucination_test_1)
print(f"\nRESULT: {'PASS' if test_1_pass else 'FAIL'}")


# TEST 2 — Give the extractor irrelevant text
irrelevant_text = """
The weather in Accra is warm and sunny today.
Temperatures may reach 31 degrees Celsius, with light winds in the afternoon.
"""

hallucination_test_2 = extract_fields(irrelevant_text)

expected_keys = {
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
}

test_2_pass = (
    isinstance(hallucination_test_2, dict)
    and set(hallucination_test_2.keys()) == expected_keys
    and hallucination_test_2.get("applicant_name") is None
    and hallucination_test_2.get("amount_ghs") is None
    and hallucination_test_2.get("monthly_profit_ghs") is None
    and hallucination_test_2.get("repayment_months") is None
    and hallucination_test_2.get("has_collateral_or_guarantor") is not True
)

print("\n" + "=" * 80)
print("TEST 2 — IRRELEVANT WEATHER TEXT")
print("=" * 80)
print(json.dumps(hallucination_test_2, indent=2))
print(f"\nRESULT: {'PASS' if test_2_pass else 'FAIL'}")

TEST 1 — MISSING CREDIT SCORE
The applicant's credit score is not provided in the loan application.

RESULT: PASS

TEST 2 — IRRELEVANT WEATHER TEXT
{
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}

RESULT: PASS


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
>
> 1. The extraction system correctly matched 15 of the 18 evaluated field values, giving an overall accuracy of 83.3%. Five fields achieved 100% accuracy, while `purpose` scored 0%. Purpose was the hardest field because the evaluator required an exact textual match. The model’s descriptions were factually similar to the gold labels but used different wording, so all three were marked as failures. This shows that exact string matching can underestimate semantic accuracy for open-text fields.
>
> 2. At temperature 0.0, all five extractions were valid JSON and identical, producing one unique result. At temperature 1.0, all five outputs were still valid JSON, but there were two unique results and the purpose wording varied in one run. This demonstrates that a higher temperature can reduce consistency even when the output remains structurally valid. A production extraction system should therefore use a low temperature and validate both its JSON structure and field values.
>
> 3. The system did not hallucinate in either probing test. When asked for L003’s unstated credit score, it correctly said that the information was not provided. When given irrelevant weather text, it returned `null` for all unavailable fields instead of inventing an applicant or financial information. Although both tests passed, hallucination risk still exists. It can be reduced through explicit “use null, do not guess” instructions, schema validation, adversarial testing, comparison with source documents, and mandatory human review.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.